In [ ]:
# CELL 1 — Imports, Reproducibility, Setup
import sys
import os
import numpy as np
import random
import tensorflow as tf

# ---------------------------------------------------------
# 1. HARDWARE & DETERMINISM CONFIGURATION
# ---------------------------------------------------------
# Force CPU-only execution. While slower, this guarantees strict 
# computational determinism by avoiding non-deterministic floating-point 
# operations inherent in GPU parallelization via cuDNN.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# Establish a global seed to ensure exact reproducibility across 
# training initializations, dropout layers, and statistical validations.
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Reproducibility settings applied (CPU mode active).")

# ---------------------------------------------------------
# 2. DEPENDENCY IMPORTS
# ---------------------------------------------------------
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import re
import time

# Spatial kinematics and signal conditioning libraries
from math import atan2, asin
from scipy.signal import butter, filtfilt

# Statistical validation and performance metric libraries
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve, 
    auc, 
    precision_recall_curve
)
from sklearn.preprocessing import label_binarize

# Deep learning architecture libraries
from tensorflow import keras
from tensorflow.keras import layers

# ---------------------------------------------------------
# 3. DIRECTORY CONFIGURATION & SYSTEM CHECKS
# ---------------------------------------------------------
# Define paths to the standardized dataset directory
RAW_ROOT = Path("IMU_Posture_Dataset_Wide_format")
CLEAN_ROOT = Path("cleaned_data_lstm")  # Directory for pre-processed signal matrices
CLEAN_ROOT.mkdir(exist_ok=True)

# Sanity checks to ensure data availability prior to pipeline execution
print("Local dataset paths set.")
print("RAW_ROOT   =", RAW_ROOT.resolve())
print("CLEAN_ROOT =", CLEAN_ROOT.resolve())

if not RAW_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset folder not found at: {RAW_ROOT.resolve()}\n"
        "Ensure the 'IMU_Posture_Dataset_Wide_format' directory is located in the "
        "same root directory as this notebook."
    )

print("Dataset ready (local environment verified).")

In [ ]:
# CELL 2 — Sensor Feature Definitions + Quaternion Math

# ---------------------------------------------------------
# 1. FEATURE SPACE CONSTRUCTION
# ---------------------------------------------------------
# Define the sensor array along the spinal axis.
SENSORS = ["L5", "T4", "C7", "T12"]

# Explicitly map the tri-axial physical kinematics to match the CSV headers
ACC_FEATURES  = ["Acceleration X(g)", "Acceleration Y(g)", "Acceleration Z(g)"]
GYRO_FEATURES = ["Angular velocity X(°/s)", "Angular velocity Y(°/s)", "Angular velocity Z(°/s)"]
QUAT_FEATURES = ["Quaternions 0()", "Quaternions 1()", "Quaternions 2()", "Quaternions 3()"]

MAG_WEAK_PREFIX = "Magnetic field"

# Dynamically map macro-posture classes from the dataset directory structure.
# This prevents hardcoding and ensures alignment with the directory names.
POSTURES = sorted([p.name for p in RAW_ROOT.iterdir() if p.is_dir()])
LABEL_TO_ID = {p: i for i, p in enumerate(POSTURES)}

print("Detected Posture Classes:", POSTURES)
print("Classification Label Map:", LABEL_TO_ID)


# ---------------------------------------------------------
# 2. SENSOR CALIBRATION UTILITIES
# ---------------------------------------------------------
def detect_mag_cols(df, sensor):
    """
    Robustly identifies tri-axial magnetometer columns. 
    Hardware logging can sometimes produce inconsistent header spacing or casing.
    This function utilizes dynamic string matching to ensure absolute data ingestion 
    without failing on subtle formatting deviations.
    """
    cols = df.columns
    out = []

    for axis in ["x", "y", "z"]:
        matches = [
            c for c in cols
            if c.lower().startswith(sensor.lower() + "_magnetic field")
            and axis in c.lower()
        ]
        if not matches:
            raise KeyError(f" Missing magnetometer axis={axis} for sensor={sensor}\n{list(cols)}")

        out.append(matches[0])

    return out


# ---------------------------------------------------------
# 3. KINEMATIC CALCULUS (ORIENTATION ENGINE)
# ---------------------------------------------------------
def quat_reorder_to_wxyz(qx, qy, qz, qw):
    """
    Standardizes quaternion arrays to the (w, x, y, z) format.
    Many hardware sensors output (x, y, z, w). Standardizing the scalar (w) 
    first ensures the downstream Hamiltonian math is computed correctly.
    """
    return np.array([qw, qx, qy, qz], dtype=np.float32)

def quat_conjugate(q):
    """
    Computes the quaternion conjugate.
    Used mathematically to represent the inverse spatial rotation.
    """
    w, x, y, z = q
    return np.array([w, -x, -y, -z], dtype=np.float32)

def quat_multiply(q1, q2):
    """
    Computes the Hamiltonian product of two quaternions.
    This enables the composition of two separate 3D rotations into a single rotation.
    """
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2
    return np.array([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2
    ], dtype=np.float32)

def rotate_vector_by_quaternion(q, v):
    """
    Applies a spatial rotation to a 3D vector using a quaternion.
    The mathematical operation is: v_rotated = q * v * q_conjugate
    """
    vq = np.array([0, v[0], v[1], v[2]], dtype=np.float32)
    return quat_multiply(quat_multiply(q, vq), quat_conjugate(q))[1:]

def quaternion_to_euler(q):
    """
    Transforms 4D quaternions into 3D Euler angles (Yaw, Pitch, Roll).
    While quaternions prevent gimbal lock during sensor tracking, Euler angles 
    provide more biomechanically interpretable features for the deep learning model 
    (e.g., pure spinal flexion maps directly to the pitch axis).
    """
    w, x, y, z = q
    
    # Mathematical conversion using standard transformation matrix logic.
    # Note: np.clip prevents domain errors in arcsin during extreme edge-case rotations.
    yaw   = atan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))
    pitch = asin(np.clip(2*(w*y - z*x), -1, 1))
    roll  = atan2(2*(w*x + y*z), 1 - 2*(x*x + y*y))
    
    return yaw, pitch, roll

In [ ]:
# CELL 3 — Preprocessing Filters

# ---------------------------------------------------------
# DIGITAL SIGNAL PROCESSING (DSP) UTILITIES
# ---------------------------------------------------------

def hampel_filter(x, window_size=5, n_sigmas=3):
    """
    Non-linear statistical outlier suppression.
    The Hampel filter slides a localized window across the time-series. 
    If a central sample deviates from the local median by more than the specified 
    number of standard deviations (estimated via Median Absolute Deviation, MAD), 
    it is classified as an impulse artifact and replaced by the median.
    
    This is highly effective for removing brief IMU connection glitches 
    without distorting the underlying kinematic waveform.
    """
    x = x.copy()
    y = x.copy()
    N = len(x)
    k = window_size

    for i in range(k, N - k):
        window = x[i - k : i + k + 1]
        median = np.nanmedian(window)
        # 1.4826 is the scale factor to make MAD an unbiased estimator of standard deviation
        mad = np.nanmedian(np.abs(window - median))

        # Prevent division-by-zero or over-filtering perfectly flat signal regions
        if mad < 1e-6:
            continue

        if abs(x[i] - median) > n_sigmas * 1.4826 * mad:
            y[i] = median

    return y


def butter_lowpass_filter(x, cutoff=3.0, fs=50, order=2):
    """
    Zero-phase low-pass digital filtering.
    A Butterworth filter is chosen for its maximally flat frequency response in the passband.
    
    Parameters:
      - cutoff (3.0 Hz): Human postural macro-movements occur at low frequencies. 
        A 3.0 Hz cutoff aggressively eliminates electrical noise and micro-tremors 
        while preserving gross spinal kinematics.
      - fs (50 Hz): The physical sampling rate of the IMU hardware.
      - filtfilt: Applies the filter forward and backward to ensure zero phase-shift, 
        preventing temporal misalignment between sensors.
    """
    # Calculate the normalized cutoff frequency (Nyquist frequency = 0.5 * fs)
    b, a = butter(order, cutoff / (0.5 * fs), btype="low")
    try:
        return filtfilt(b, a, x, axis=0)
    except:
        # Fallback to returning raw data if the signal array is too short for the filter order
        return x


def remove_bias(x):
    """
    DC offset mitigation (Zero-mean centering).
    Subtracts the global mean from the signal to mitigate static sensor calibration 
    drift. This ensures the deep learning model evaluates the dynamic variations 
    rather than arbitrary absolute initial values.
    """
    return x - np.nanmean(x, axis=0)

In [ ]:
# CELL 4 — Full Preprocessing Pipeline
def preprocess_single_file(raw_path):
    """
    Ingests and transforms a single raw experimental trial (CSV) into a fully 
    conditioned, spatially aligned, and temporally synchronized feature matrix.
    
    Parameters:
      raw_path (Path): Filepath to the specific trial within the dataset.
    Returns:
      X_out (np.ndarray): The continuous, preprocessed multichannel kinematic matrix.
    """
    df = pd.read_csv(raw_path)

    # ---------------------------------------------------------
    # 1. CHANNEL SELECTION & DATA INTEGRITY
    # ---------------------------------------------------------
    # Dynamically extract only the required physical kinematic columns 
    # to exclude irrelevant hardware metadata or timestamp columns.
    keep = []
    for col in df.columns:
        for s in SENSORS:
            if col.startswith(f"{s}_Acceleration"):     keep.append(col)
            elif col.startswith(f"{s}_Angular velocity"): keep.append(col)
            elif col.startswith(f"{s}_Quaternions"):     keep.append(col)
            elif col.startswith(f"{s}_Magnetic field"):  keep.append(col)

    # Coerce to numeric format. If hardware errors wrote string characters (e.g., "ERR"), 
    # this forces them to NaN for downstream imputation.
    df = df[keep].copy().apply(pd.to_numeric, errors="coerce")

    # Handle transient wireless packet loss (NaN/Inf values)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Linear interpolation reconstructs short temporal gaps in the IMU transmission 
    # without distorting the overall macro-postural waveform.
    df.interpolate(limit_direction="both", inplace=True)
    
    # Forward and backward fill act as fail-safes for missing data at the absolute 
    # beginning or end of the trial where interpolation cannot anchor.
    df.ffill(inplace=True)
    df.bfill(inplace=True)

    blocks = []

    # ---------------------------------------------------------
    # 2. SENSOR-LEVEL KINEMATIC CONDITIONING
    # ---------------------------------------------------------
    for sensor in SENSORS:
        # Extract raw arrays for the current spinal location
        acc = df[[f"{sensor}_{f}" for f in ACC_FEATURES]].values
        gyr = df[[f"{sensor}_{f}" for f in GYRO_FEATURES]].values
        quat_raw = df[[f"{sensor}_{f}" for f in QUAT_FEATURES]].values
        mag_cols = detect_mag_cols(df, sensor)
        mag = df[mag_cols].values

        # Quaternion Normalization
        # Quaternions must have a magnitude (L2 norm) of exactly 1.0 to represent 
        # a valid pure rotation. Hardware drift can cause slight denormalization.
        quats = []
        for (qx, qy, qz, qw) in quat_raw:
            q = quat_reorder_to_wxyz(qx, qy, qz, qw)
            q = q / np.linalg.norm(q)
            quats.append(q)
        quats = np.array(quats)

        # Apply DC Bias mitigation (Zero-mean centering)
        acc = remove_bias(acc)
        gyr = remove_bias(gyr)
        mag = remove_bias(mag)

        # Apply non-linear Hampel filtering (per axis) to suppress impulse artifacts
        for i in range(acc.shape[1]): acc[:, i] = hampel_filter(acc[:, i])
        for i in range(gyr.shape[1]): gyr[:, i] = hampel_filter(gyr[:, i])
        for i in range(mag.shape[1]): mag[:, i] = hampel_filter(mag[:, i])

        # Apply zero-phase Butterworth low-pass filtering to isolate human kinematics
        acc = butter_lowpass_filter(acc)
        gyr = butter_lowpass_filter(gyr)
        mag = butter_lowpass_filter(mag)

        # ---------------------------------------------------------
        # 3. SPATIAL ALIGNMENT & FEATURE ENGINEERING
        # ---------------------------------------------------------
        A, G, M, E = [], [], [], []

        for t in range(len(df)):
            q = quats[t]
            
            # Rotate local vectors into the absolute global reference frame.
            # This mathematically compensates for slight variations in how the 
            # Velcro straps were attached to different subjects' backs.
            A.append(rotate_vector_by_quaternion(q, acc[t]))
            G.append(rotate_vector_by_quaternion(q, gyr[t]))
            M.append(rotate_vector_by_quaternion(q, mag[t]))
            
            # Extract interpretable Biomechanical Euler Angles (Yaw, Pitch, Roll)
            E.append(quaternion_to_euler(q))

        A = np.array(A); G = np.array(G); M = np.array(M); E = np.array(E)

        # Compute L2 Norm Magnitudes
        # Magnitudes provide the CNN with rotation-invariant measures of kinematic 
        # intensity, improving generalizability.
        A_mag = np.linalg.norm(A, axis=1, keepdims=True)
        G_mag = np.linalg.norm(G, axis=1, keepdims=True)
        M_mag = np.linalg.norm(M, axis=1, keepdims=True)

        # Fuse all engineered features for this specific sensor into a single block
        # Shape per row: 3(A) + 3(G) + 3(M) + 1(A_mag) + 1(G_mag) + 1(M_mag) + 3(Euler) = 15 features per sensor
        block = np.concatenate([A, G, M, A_mag, G_mag, M_mag, E], axis=1)
        blocks.append(block)

    # ---------------------------------------------------------
    # 4. CROSS-SENSOR AGGREGATION
    # ---------------------------------------------------------
    # Concatenate all 4 sensors horizontally (15 features * 4 sensors = 60 feature columns)
    X_out = np.concatenate(blocks, axis=1)

    return X_out

In [ ]:
# CELL 5 — Load and Preprocess All Data
print("\n===== PREPROCESSING ALL FILES =====\n")

# ---------------------------------------------------------
# 1. METADATA EXTRACTION CONFIGURATION
# ---------------------------------------------------------
# Regular expression to extract the Subject ID and Trial ID from the filename.
# Expected format: "Sub_1_Trial_1.csv"
# Group 1: Subject ID (\d+)
# Group 2: Trial ID (\d+)
SUBJECT_RE = re.compile(r"Sub_(\d+)_Trial_(\d+)\.csv")

rows = []

# ---------------------------------------------------------
# 2. BATCH PROCESSING LOOP
# ---------------------------------------------------------
for posture_dir in RAW_ROOT.iterdir():
    # Ignore any stray files in the root directory; only parse subdirectories
    if not posture_dir.is_dir():
        continue

    files = list(posture_dir.glob("*.csv"))
    print(f"Processing posture class: {posture_dir.name} ({len(files)} files)")

    for raw_file in files:
        # Execute the end-to-end DSP and spatial alignment pipeline.
        # This is performed entirely in-memory to eliminate I/O bottlenecking 
        # from writing intermediate filtered CSVs to disk.
        X_processed = preprocess_single_file(raw_file)
        
        # Parse the filename to enforce subject-level data isolation
        m = SUBJECT_RE.search(raw_file.name)
        if m:
            rows.append({
                "data": X_processed,                     # The conditioned 2D kinematic matrix (Time x Features)
                "posture": posture_dir.name,             # String representation of the macro-posture
                "label": LABEL_TO_ID[posture_dir.name],  # Integer class label for cross-entropy loss
                "subject": int(m.group(1)),              # Subject ID (Critical for LOSO cross-validation)
                "trial": int(m.group(2)),                # Trial ID (For granular performance auditing)
                "filename": raw_file.name                # Original reference filename
            })

# ---------------------------------------------------------
# 3. CORPUS ENCAPSULATION & VERIFICATION
# ---------------------------------------------------------
# Encapsulate the entire processed dataset into a structured Pandas DataFrame 
# to facilitate efficient querying and splitting during the LOSO evaluation.
df_items = pd.DataFrame(rows)

print(f"\nTotal processed files: {len(df_items)}")
print(f"Unique subjects detected: {df_items['subject'].nunique()}")
print(f"Engineered feature dimension per temporal frame: {df_items['data'].iloc[0].shape[1]}")

In [ ]:
# CELL 6 — Windowing

# ---------------------------------------------------------
# 1. TEMPORAL RESOLUTION CONFIGURATION
# ---------------------------------------------------------
FS = 50          # Hardware sampling frequency (Hz)
WIN_LEN = 200    # Window size: 200 frames = 4.0 seconds of continuous kinematics
STRIDE = 100     # Sliding stride: 100 frames = 50% temporal overlap

print(f"\nWindow length = {WIN_LEN} frames ({WIN_LEN/FS:.2f} sec)")
print(f"Stride = {STRIDE} frames (50% overlap)")

# ---------------------------------------------------------
# 2. SLIDING WINDOW ENGINE
# ---------------------------------------------------------
def make_windows(X):
    """
    Transforms a continuous 2D kinematic matrix into a 3D tensor of discrete windows.
    
    Parameters:
      X (np.ndarray): Continuous trial data of shape (T, F), where T is total frames 
                      and F is the engineered feature count (e.g., 60).
    
    Returns:
      np.ndarray: A 3D tensor of shape (num_windows, WIN_LEN, F) ready for 
                  1D-Convolutional ingestion.
    """
    T, F = X.shape

    # ---------------------------------------------------------
    # Edge-Case Handling: Zero-Padding
    # ---------------------------------------------------------
    # If an experimental trial was prematurely terminated and contains fewer frames 
    # than the requisite window length, pad the remainder with zeros to maintain 
    # the strict dimensionality required by the CNN tensor inputs.
    if T < WIN_LEN:
        w = np.zeros((WIN_LEN, F))
        w[:T] = X
        return w[None, :, :]

    # ---------------------------------------------------------
    # Window Extraction
    # ---------------------------------------------------------
    windows = []
    # Slide across the time axis (T), stepping by the specified STRIDE.
    # The +1 ensures the loop bounds capture the final valid window perfectly.
    for i in range(0, T - WIN_LEN + 1, STRIDE):
        windows.append(X[i : i + WIN_LEN])

    return np.array(windows)

In [ ]:
# CELL 7 — Create Window Dataset (ACC + EULER ONLY)

# ---------------------------------------------------------
# 1. TENSOR INITIALIZATION
# ---------------------------------------------------------
# Pre-allocate lists to construct the final dataset arrays.
# We explicitly track subject and trial IDs alongside the data (X) and labels (y)
# to strictly enforce the Leave-One-Subject-Out (LOSO) boundaries later.
X_all = []
y_all = []
subj_all = []
trial_all = []

window_counter = 0

# ---------------------------------------------------------
# 2. FEATURE SELECTION & SEGMENTATION LOOP
# ---------------------------------------------------------
for _, row in df_items.iterrows():

    X_full = row["data"]  # Extract the 60-feature preprocessed matrix from memory

    # ---------------------------------------------------------
    # Targeted Feature Slicing: ACC + EULER ONLY
    # ---------------------------------------------------------
    # The baseline CNN utilizes a 24-feature subset. We discard Angular Velocity 
    # (since the postures are static holds) and Magnetometer (to avoid magnetic 
    # distortion artifacts in clinical/office environments).
    #
    # Block mapping from the 60-feature matrix (15 features per sensor):
    # L5  (Indices 0-14):  Acc (0-2)   | Euler (12-14)
    # T4  (Indices 15-29): Acc (15-17) | Euler (27-29)
    # C7  (Indices 30-44): Acc (30-32) | Euler (42-44)
    # T12 (Indices 45-59): Acc (45-47) | Euler (57-59)
    # ---------------------------------------------------------
    acc_euler_slices = [
        np.r_[0:3, 12:15],    # L5 Subset
        np.r_[15:18, 27:30],  # T4 Subset
        np.r_[30:33, 42:45],  # C7 Subset
        np.r_[45:48, 57:60]   # T12 Subset
    ]

    # Horizontally concatenate the selected slices to form the 24-feature matrix
    X = np.concatenate([X_full[:, s] for s in acc_euler_slices], axis=1)

    # Transform the continuous 2D matrix into discrete 3D overlapping windows
    windows = make_windows(X)

    # Append the newly formed windows and their corresponding metadata
    for w in windows:
        X_all.append(w)
        y_all.append(row["label"])
        subj_all.append(row["subject"])
        trial_all.append(row["trial"])

    window_counter += len(windows)

# ---------------------------------------------------------
# 3. FINAL ARRAY CONVERSION
# ---------------------------------------------------------
# Convert Python lists to optimal NumPy arrays for TensorFlow ingestion.
X_all = np.array(X_all)
y_all = np.array(y_all)
subj_all = np.array(subj_all)
trial_all = np.array(trial_all)

# ---------------------------------------------------------
# 4. DATASET VERIFICATION AUDIT
# ---------------------------------------------------------
print("\n===== DATASET SUMMARY =====")
print("Total discrete temporal windows:", window_counter)
# Expected Shape: (N_windows, 200 temporal frames, 24 spatial features)
print("Final Input Tensor (X) shape:", X_all.shape)  
print("Targeted Features per timestep:", X_all.shape[2])

In [ ]:
# CELL 8 — Normalization (ACC + EULER → 6 per sensor)

# ---------------------------------------------------------
# 1. SENSOR CHANNEL MAPPING
# ---------------------------------------------------------
# Define the exact column slices for the 24-feature input tensor.
# Each of the 4 sensors (L5, T4, C7, T12) contributes exactly 6 features 
# (3 Acceleration + 3 Euler Angles).
SENSOR_SLICES = [
    slice(0, 6),   # Sensor 1 (L5)
    slice(6, 12),  # Sensor 2 (T4)
    slice(12, 18), # Sensor 3 (C7)
    slice(18, 24)  # Sensor 4 (T12)
]

# ---------------------------------------------------------
# 2. STATISTICAL MOMENT COMPUTATION
# ---------------------------------------------------------
def compute_sensor_norm(X):
    """
    Computes the Mean and Standard Deviation for each sensor independently 
    across a given data split (typically the Training fold).
    
    Parameters:
      X (np.ndarray): 3D tensor of shape (N_windows, 200, 24).
    
    Returns:
      means (list): Computed means for each of the 4 sensors.
      stds (list): Computed standard deviations for each of the 4 sensors.
    """
    means = []
    stds  = []

    for s in SENSOR_SLICES:
        # Isolate the 6 features belonging to the current sensor across all windows
        sensor_data = X[:, :, s]
        
        # Flatten the temporal dimension to compute global statistics for the sensor 
        # rather than window-specific or timestep-specific statistics.
        flat = sensor_data.reshape(-1, sensor_data.shape[-1])

        # Compute statistical moments, preserving the feature axis dimension
        mean = flat.mean(axis=0, keepdims=True)
        # Add a microscopic epsilon (1e-8) to prevent ZeroDivisionError in cases 
        # where a feature exhibits absolutely zero variance.
        std  = flat.std(axis=0, keepdims=True) + 1e-8

        means.append(mean)
        stds.append(std)

    return means, stds


# ---------------------------------------------------------
# 3. Z-SCORE APPLICATION
# ---------------------------------------------------------
def apply_sensor_norm(X, means, stds):
    """
    Applies Z-score standardization using pre-computed statistics.
    By accepting externally computed means and stds, this function safely 
    normalizes validation and testing data without leaking their internal distributions.
    
    Parameters:
      X (np.ndarray): 3D tensor to be normalized.
      means (list): Pre-computed means from the training set.
      stds (list): Pre-computed standard deviations from the training set.
      
    Returns:
      Xn (np.ndarray): The standardized 3D tensor.
    """
    # Create an explicit copy to prevent unintended in-place mutation of the raw array
    Xn = X.copy()

    for s, mean, std in zip(SENSOR_SLICES, means, stds):
        # Apply standard Z-score formula: z = (x - μ) / σ
        Xn[:, :, s] = (X[:, :, s] - mean) / std

    return Xn

In [ ]:
# CELL 9 — SensorDropout Layer
class SensorDropout(layers.Layer):
    """
    Custom Keras Layer for Structured Sensor-Level Dropout.
    Dynamically deactivates entire IMU sensor feature blocks during training 
    to force the network to learn robust, distributed kinematic representations.
    """

    def __init__(self, drop_prob=0.2):
        super().__init__()
        # drop_prob represents the probability that any given sensor (out of 4) 
        # is entirely zeroed out during a training step. Default is 20%.
        self.drop_prob = drop_prob

    def call(self, x, training=None):
        """
        Forward pass for the custom dropout layer.
        
        Parameters:
          x (Tensor): Input tensor of shape (batch, timesteps, features).
          training (bool): Keras flag indicating whether the model is in training 
                           or inference (evaluation/testing) mode.
        """
        # Dropout is strictly a training-phase regularization technique. 
        # During validation/LOSO testing, all sensors must pass data cleanly.
        if not training:
            return x

        # Dynamically extract tensor dimensions
        batch = tf.shape(x)[0]
        n_sensors = 4
        
        # Calculate features per sensor (e.g., 24 total features // 4 sensors = 6)
        feat_per_sensor = tf.shape(x)[-1] // n_sensors

        # Generate a binary mask for the sensors.
        # Shape: (batch, 4 sensors, 1). 
        # A uniform random float is drawn; if it is greater than the drop_prob, 
        # the mask is 1 (keep sensor). If less, the mask is 0 (drop sensor).
        mask = tf.cast(
            tf.random.uniform((batch, n_sensors, 1)) > self.drop_prob,
            x.dtype
        )

        # Broadcast the mask across the feature dimension.
        # This ensures that if a sensor is dropped, ALL of its specific features 
        # (Acc X, Y, Z + Euler Yaw, Pitch, Roll) are zeroed out together.
        mask = tf.repeat(mask, repeats=feat_per_sensor, axis=2)
        
        # Reshape to match the input tensor for element-wise multiplication:
        # Shape becomes (batch, 1, 24 features).
        # The '1' in the temporal dimension allows TensorFlow to automatically 
        # broadcast the mask across all 200 temporal frames of the window.
        mask = tf.reshape(mask, (batch, 1, n_sensors * feat_per_sensor))

        # Apply the mask to the input tensor
        return x * mask

In [ ]:
# CELL 10 — Model with Attention Extraction
def build_model(input_shape):
    """
    Constructs the baseline Convolutional Neural Network (CNN) with hierarchical attention.
    
    Parameters:
      input_shape (tuple): Expected shape of a single temporal window (e.g., (200, 24)).
    Returns:
      model (keras.Model): The compiled Keras model ready for LOSO training.
    """
    inp = keras.Input(shape=input_shape)

    print("Model input shape:", input_shape)

    # 1. HARDWARE REGULARIZATION
    # Apply custom SensorDropout (defined in Cell 9) with a 25% drop probability 
    # to encourage cross-sensor generalization during training.
    x = SensorDropout(0.25)(inp)

    # ---------------------------------------------------------
    # 2. PARALLEL SENSOR SPLITTING
    # ---------------------------------------------------------
    # Isolate the 24-channel tensor back into 4 independent 6-channel streams.
    # This forces the initial convolutional layers to learn local spatial-temporal 
    # patterns *within* a specific anatomical location before global fusion.
    L5  = layers.Lambda(lambda x: x[:, :, 0:6])(x)
    T4  = layers.Lambda(lambda x: x[:, :, 6:12])(x)
    C7  = layers.Lambda(lambda x: x[:, :, 12:18])(x)
    T12 = layers.Lambda(lambda x: x[:, :, 18:24])(x)

    print("Sensor feature size per parallel stream:", 6)

    # ---------------------------------------------------------
    # 3. CONVOLUTIONAL FEATURE EXTRACTION & ATTENTION
    # ---------------------------------------------------------
    def sensor_block(x, name_prefix):
        """
        Processes a single sensor stream through successive 1D convolutions 
        and applies feature-level attention.
        """
        # Block 1: Temporal extraction (Window size = 5)
        x = layers.Conv1D(32, 5, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x) # Mitigates internal covariate shift

        # Block 2: Finer temporal extraction (Window size = 3)
        x = layers.Conv1D(32, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)

        # Feature-Level Attention Mechanism
        # Learns to dynamically emphasize highly discriminative convolutional filters 
        # while suppressing noisy or irrelevant ones.
        attention = layers.Dense(32, activation="tanh", name=f"{name_prefix}_feat_dense")(x)
        attention = layers.Softmax(axis=-1, name=f"{name_prefix}_feat_softmax")(attention)

        # Apply the learned attention weights to the extracted features
        x = layers.Multiply(name=f"{name_prefix}_feat_mul")([x, attention])

        return x, attention

    # Pass each isolated sensor stream through its own convolutional block
    L5,  L5_feat_att  = sensor_block(L5,  "L5")
    T4,  T4_feat_att  = sensor_block(T4,  "T4")
    C7,  C7_feat_att  = sensor_block(C7,  "C7")
    T12, T12_feat_att = sensor_block(T12, "T12")

    # ---------------------------------------------------------
    # 4. SENSOR-LEVEL ATTENTION & FUSION
    # ---------------------------------------------------------
    # Re-stack the 4 processed streams. Shape becomes: (batch, timesteps, 4, 32)
    sensors = layers.Lambda(lambda x: tf.stack(x, axis=2))([L5, T4, C7, T12])

    # Sensor-Level Attention Mechanism
    # Learns which anatomical sensor is providing the most reliable kinematic 
    # data for the current posture at any given timestep.
    sensor_attention = layers.Dense(1, activation="tanh", name="sensor_dense")(sensors)
    sensor_attention = layers.Softmax(axis=2, name="sensor_softmax")(sensor_attention)

    # Apply sensor weights
    sensors = layers.Multiply()([sensors, sensor_attention])

    # Feature Fusion: Collapse the sensor axis by summing the weighted streams,
    # returning the tensor to a 3D shape: (batch, timesteps, 32)
    fused = layers.Lambda(lambda x: tf.reduce_sum(x, axis=2))(sensors)

    # ---------------------------------------------------------
    # 5. TEMPORAL POOLING & CLASSIFICATION
    # ---------------------------------------------------------
    # Global Average Pooling (GAP) averages the 32 features across all 200 timesteps.
    # This aggressively reduces dimensionality, preventing overfitting compared to Flatten().
    x = layers.GlobalAveragePooling1D()(fused)

    # Fully Connected Head with aggressive dropout for regularization
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    # Output layer: Maps to the distinct posture classes
    out = layers.Dense(len(POSTURES), activation="softmax")(x)

    model = keras.Model(inp, out)

    # ---------------------------------------------------------
    # 6. OPTIMIZATION & COMPILATION
    # ---------------------------------------------------------
    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        # Label Smoothing (0.05) softens the strict one-hot encoded targets (1.0 vs 0.0).
        # This acts as a regularizer, preventing the model from becoming overly confident 
        # and heavily penalizing misclassifications in biomechanically similar classes 
        # (e.g., Slouching vs. Forward Bending).
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"]
    )

    # ---------------------------------------------------------
    # 7. METADATA ATTACHMENT FOR INTERPRETABILITY
    # ---------------------------------------------------------
    # Dynamically bind the attention tensors to the Keras model object.
    # This allows us to extract and visualize the attention maps post-training 
    # to understand what the model is biologically "focusing" on.
    model.feature_attentions = [L5_feat_att, T4_feat_att, C7_feat_att, T12_feat_att]
    model.sensor_attention = sensor_attention

    model.summary()

    return model

In [ ]:
# CELL 11 — Learning Rate Scheduler

# ---------------------------------------------------------
# ADAPTIVE GRADIENT DESCENT SCHEDULING
# ---------------------------------------------------------
lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    # Monitor the validation loss (rather than training loss) to ensure the 
    # decay is triggered by plateauing generalization, not just memorization.
    monitor="val_loss",
    
    # Decay Factor: If triggered, the current learning rate is multiplied by 0.5 (halved).
    # This allows the optimizer to take finer steps into the loss gradient minima.
    factor=0.5,
    
    # Patience: The number of epochs with no improvement before triggering the decay.
    # A patience of 3 absorbs minor epoch-to-epoch statistical noise without 
    # waiting so long that computational time is wasted.
    patience=3,
    
    # Minimum Learning Rate: Establishes a hard floor (1e-6) to prevent the 
    # optimizer from decaying the learning rate to zero, which would effectively 
    # freeze the network weights and halt all learning.
    min_lr=1e-6,
    
    # Verbosity: Prints a notification to the standard output when a decay event 
    # occurs, which is highly useful for auditing the training dynamics.
    verbose=1
)

In [ ]:
# CELL 12 — LOSO Loop (Per-Fold Only)
from collections import defaultdict
import gc

# ---------------------------------------------------------
# 1. COHORT COHESION & FILTERING
# ---------------------------------------------------------
# Exclude specific subjects who may have displayed anomalous movement behaviors, 
# hardware calibration errors, or incomplete logging trials during collection.
EXCLUDED_SUBJECTS = [16, 21, 26, 29]

# Extract the validated subject cohort
subjects = sorted(
    s for s in np.unique(subj_all)
    if s not in EXCLUDED_SUBJECTS
)

# Initialize arrays to monitor performance tracking metrics across folds
window_scores = []
trial_scores = []

all_window_true = []
all_window_pred = []

all_trial_true = []
all_trial_pred = []
all_trial_subjects = []

# Accumulate test probabilities and raw inputs for post-hoc attention maps
all_window_probs = []
all_attention_inputs = []   

fold_results = []

# ---------------------------------------------------------
# 2. LOSO CROSS-VALIDATION CORE EXECUTION
# ---------------------------------------------------------
for test_subject in subjects:

    print("\n============================")
    print(f"LOSO EVALUATION FOLD - TEST SUBJECT: {test_subject}")
    print("============================")

    # -----------------------------------------------------
    # A. SUBJECT-LEVEL TRAIN/VAL/TEST SEGREGATION
    # -----------------------------------------------------
    # Isolate all remaining individuals for the training matrix
    train_subjects = [s for s in subjects if s != test_subject]

    # Enforce strict validation isolation: Allocate 20% of the training cohort 
    # to serve as an unseen validation monitor for early stopping parameters.
    val_size = max(1, int(0.2 * len(train_subjects)))
    val_subjects = np.random.choice(train_subjects, size=val_size, replace=False)
    train_subjects_final = [s for s in train_subjects if s not in val_subjects]

    # Generate boolean indexing masks to preserve subject-level boundaries
    train_mask = np.isin(subj_all, train_subjects_final)
    val_mask   = np.isin(subj_all, val_subjects)
    test_mask  = subj_all == test_subject

    # Partition the main input tensors
    X_train = X_all[train_mask]
    y_train = y_all[train_mask]

    X_val   = X_all[val_mask]
    y_val   = y_all[val_mask]

    X_test   = X_all[test_mask]
    y_test   = y_all[test_mask]

    # Track structural metadata tags for trial aggregation
    trial_test = trial_all[test_mask]
    subject_test = subj_all[test_mask]

    # -----------------------------------------------------
    # B. Z-SCORE NORMALIZATION WITH LEAKAGE PREVENTION
    # -----------------------------------------------------
    # Compute statistical moments ONLY from the current training cohort fold
    means, stds = compute_sensor_norm(X_train)

    # Standardize splits using the training-derived calibration statistics
    X_train = apply_sensor_norm(X_train, means, stds)
    X_val   = apply_sensor_norm(X_val, means, stds)
    X_test  = apply_sensor_norm(X_test, means, stds)

    # Cache the standardized test input for down-stream attention matrix rendering
    all_attention_inputs.append(X_test)

    # -----------------------------------------------------
    # C. ONE-HOT CATEGORICAL ENCODING
    # -----------------------------------------------------
    y_train_cat = keras.utils.to_categorical(y_train, len(POSTURES))
    y_val_cat   = keras.utils.to_categorical(y_val, len(POSTURES))

    # -----------------------------------------------------
    # D. NETWORK COMPILATION & OPTIMIZATION
    # -----------------------------------------------------
    model = build_model(X_train.shape[1:])

    # Early stopping limits overfitting by halting training when validation loss 
    # fails to decline for 5 consecutive epochs, resetting to optimal parameters.
    early = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    # Execute training pass for the isolated fold
    hist = model.fit(
        X_train,
        y_train_cat,
        validation_data=(X_val, y_val_cat),
        epochs=60,
        batch_size=32,
        callbacks=[early, lr_scheduler],
        verbose=1
    )

    # -----------------------------------------------------
    # E. LEVEL 1: WINDOW-LEVEL DISCRIMINATION AUDIT
    # -----------------------------------------------------
    # Generate continuous probabilities across all classes for the test subject
    probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    y_true = y_test

    all_window_probs.append(probs)

    # Calculate window performance metric
    window_acc = np.mean(y_pred == y_true)
    window_scores.append(window_acc)

    print(f"\nWINDOW ACCURACY FOR SUBJECT {test_subject}: {window_acc:.4f}")
    print(classification_report(y_true, y_pred, target_names=POSTURES))

    # Render localized confusion matrices
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=POSTURES).plot()
    plt.title(f"Window CM - Subject {test_subject}")
    plt.show()

    all_window_true.extend(y_true)
    all_window_pred.extend(y_pred)

    # -----------------------------------------------------
    # F. LEVEL 2: TRIAL-LEVEL LOG-LIKELIHOOD FUSION
    # -----------------------------------------------------
    # Map windows back to their original distinct trials using a hashable key
    trial_probs = defaultdict(list)
    trial_true = {}

    for p, true, s, t in zip(probs, y_true, subject_test, trial_test):
        key = (s, t)
        trial_probs[key].append(p)
        trial_true[key] = true

    trial_pred = []
    trial_true_list = []
    trial_subjects = []

    # Enforce Bayesian log-likelihood combination across trial components
    for k in trial_probs:
        w = np.array(trial_probs[k])
        # Summing log probabilities instead of multiplying raw weights prevents 
        # numerical underflow and creates a highly stable ensemble vote.
        summed = np.sum(np.log(w + 1e-8), axis=0)
        pred = np.argmax(summed)

        trial_pred.append(pred)
        trial_true_list.append(trial_true[k])
        trial_subjects.append(k[0])

    trial_pred = np.array(trial_pred)
    trial_true_list = np.array(trial_true_list)
    trial_subjects = np.array(trial_subjects)

    # Calculate overarching trial macro performance
    trial_acc = np.mean(trial_pred == trial_true_list)
    trial_scores.append(trial_acc)

    print(f"\nTRIAL-LEVEL (FUSED) ACCURACY FOR SUBJECT {test_subject}: {trial_acc:.4f}")
    print(classification_report(trial_true_list, trial_pred, target_names=POSTURES))

    cm = confusion_matrix(trial_true_list, trial_pred)
    ConfusionMatrixDisplay(cm, display_labels=POSTURES).plot()
    plt.title(f"Trial CM - Subject {test_subject}")
    plt.show()

    all_trial_true.extend(trial_true_list)
    all_trial_pred.extend(trial_pred)
    all_trial_subjects.extend(trial_subjects)

    # Cache parameters of the final loop fold to provide validation references
    if test_subject == subjects[-1]:
        final_model = model
        final_means = means
        final_stds = stds

    # -----------------------------------------------------
    # G. RESOURCE CLEANUP
    # -----------------------------------------------------
    # Explicitly clear graphs and run garbage collector to prevent memory leaks 
    # across successive training loops.
    del model, hist
    gc.collect()

In [ ]:
# CELL 13 — Final Results (All Folds)
print("\n===== FINAL LOSO RESULTS =====")

# Output global cross-validated metrics across the cohort
print("Avg Window Acc:", np.mean(window_scores))
print("Avg Trial Acc:", np.mean(trial_scores))

# =========================================================
# 1. TENSOR CONCATENATION & SHAPE ALIGNMENT
# =========================================================
# Flatten the nested lists of probability arrays accumulated across all independent folds 
# into a contiguous 2D array for global evaluation.
all_window_probs = np.concatenate(all_window_probs, axis=0)

print("Probs shape:", all_window_probs.shape)
print("True shape:", np.array(all_window_true).shape)

# ---------------------------------------------------------
# 2. TIER 1: WINDOW-LEVEL METRICS (MICRO EVALUATION)
# ---------------------------------------------------------
# Assesses the model's ability to classify posture instantaneously within a 4-second window.
print("\nWINDOW METRICS")
print(classification_report(all_window_true, all_window_pred, target_names=POSTURES))

cm = confusion_matrix(all_window_true, all_window_pred)
ConfusionMatrixDisplay(cm, display_labels=POSTURES).plot()
plt.title("Window CM (LOSO)")
plt.show()

# ---------------------------------------------------------
# 3. TIER 2: TRIAL-LEVEL METRICS (MESO EVALUATION)
# ---------------------------------------------------------
# Assesses performance after log-likelihood temporal smoothing across full continuous sequences.
print("\nTRIAL METRICS")
print(classification_report(all_trial_true, all_trial_pred, target_names=POSTURES))

cm = confusion_matrix(all_trial_true, all_trial_pred)
ConfusionMatrixDisplay(cm, display_labels=POSTURES).plot()
plt.title("Trial CM (LOSO)")
plt.show()

# =========================================================
# 4. TIER 3: SUBJECT-LEVEL METRICS (MACRO VOTING EVALUATION)
# =========================================================
# Aggregates decisions across all completed tasks for a specific individual. 
# This determines if the classifier remains robust for a user over long-term monitoring.
subject_votes = defaultdict(list)
subject_true = {}

# Map the smoothed trial-level predictions back to their respective subject groupings
for pred, true, s in zip(all_trial_pred, all_trial_true, all_trial_subjects):
    key = (s, true)
    subject_votes[key].append(pred)
    subject_true[key] = true

all_subject_pred = []
all_subject_true = []

# Apply a majority voting (plurality) consensus rule across all of a subject's trials.
# argmax(bincount) extracts the most frequently occurring predicted posture class.
for k in subject_votes:
    votes = np.array(subject_votes[k])
    pred = np.bincount(votes).argmax()

    all_subject_pred.append(pred)
    all_subject_true.append(subject_true[k])

all_subject_pred = np.array(all_subject_pred)
all_subject_true = np.array(all_subject_true)

print("\n===== SUBJECT RESULTS =====")
print(f"Avg Subject Acc: {np.mean(all_subject_pred == all_subject_true):.4f}")

# ---------------------------------------------------------
# 5. SUBJECT-LEVEL CLASSIFICATION REPORT & VISUALIZATION
# ---------------------------------------------------------
print("\nSUBJECT METRICS")
print(classification_report(
    all_subject_true,
    all_subject_pred,
    labels=np.arange(len(POSTURES)),
    target_names=POSTURES,
    zero_division=0 # Prevents undefined metric warnings if a class is entirely unpredicted
))

cm = confusion_matrix(
    all_subject_true,
    all_subject_pred,
    labels=np.arange(len(POSTURES))
)

ConfusionMatrixDisplay(cm, display_labels=POSTURES).plot()
plt.title("Subject Confusion Matrix (LOSO)")
plt.show()